In [ ]:
!pip uninstall -y paddleocr paddlepaddle paddlepaddle-gpu paddlex 2>/dev/null
!pip install --force-reinstall --no-cache-dir \
    paddleocr==2.8.0 \
    paddlepaddle-gpu==2.6.2 \
    -f https://www.paddlepaddle.org.cn/whl/linux/gpu.html

In [1]:
import os, json, time, cv2, numpy as np
from pathlib import Path

In [2]:
os.environ["FLAGS_allocator_strategy"] = "auto_growth"  
os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.6"  
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
from concurrent.futures import ThreadPoolExecutor
from paddleocr import PaddleOCR

IMAGE_FOLDER = "/kaggle/input/datasets/shpk512/ttttttt/images"
OUTPUT_FOLDER = "/kaggle/working/ocr_results"
CONF_THRESHOLD = 0.55  

ocr = PaddleOCR(
    lang='en',
    use_angle_cls=True,      
    use_gpu=True,
    det_limit_side_len=1536, 
    det_db_thresh=0.3,       
    det_db_box_thresh=0.5,   
    det_db_unclip_ratio=1.6, 
    rec_batch_num=8,
    show_log=False
)
print("Model is ready")

download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 4.00M/4.00M [00:26<00:00, 151kiB/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10.2M/10.2M [00:03<00:00, 3.38MiB/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:16<00:00, 135kiB/s] 


Model is ready


In [ ]:
print("\nTest first 5 images")
for check_path in image_paths[:5]:
    res = ocr.ocr(check_path, cls=True)
    img = cv2.imread(check_path)
    count = 0
    if res and res[0]:
        for line in res[0]:
            if len(line) >= 2 and line[1][1] >= CONF_THRESHOLD:
                pts = np.array(line[0], dtype=np.int32)
                cv2.polylines(img, [pts], True, (0, 255, 0), 2)
                text = f"{line[1][0]} ({line[1][1]:.2f})"
                cv2.putText(img, text, (pts[0][0], pts[0][1]-8), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                count += 1
    out_name = f"debug_{Path(check_path).stem}.jpg"
    cv2.imwrite(os.path.join(OUTPUT_FOLDER, out_name), img)
    print(f"{Path(check_path).name}: find {count} lines (save in {out_name})")


In [5]:
import paddle
image_paths = [os.path.join(IMAGE_FOLDER, f) for f in os.listdir(IMAGE_FOLDER) 
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
total = len(image_paths)
print(f"Find images: {total}")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
results = []
start_time = time.time()
print(f"\nRunning OCR")
for i, img_path in enumerate(image_paths):
    try:
        res = ocr.ocr(img_path, cls=True)
    except Exception as e:
        error_msg = str(e)
        if "Out of memory" in error_msg or "ResourceExhaustedError" in error_msg:
            print(f"\nOOM on {os.path.basename(img_path)}. Try again")
            paddle.device.cuda.empty_cache()
            time.sleep(0.5)
            try:
                res = ocr.ocr(img_path, cls=True)
            except Exception as e2:
                print(f"OOM. Skip file.")
                results.append({"image": os.path.basename(img_path), "detections": [], "count": 0, "error": str(e2)})
                continue
        else:
            results.append({"image": os.path.basename(img_path), "detections": [], "count": 0, "error": str(e)})
            continue
    detections = []
    if res and res[0]:
        for line in res[0]:
            if len(line) >= 2 and line[1][1] >= CONF_THRESHOLD:
                polygon = line[0]
                detections.append({
                    "bbox": {
                        "polygon": [[int(p[0]), int(p[1])] for p in polygon],
                    },
                    "text": str(line[1][0]),
                    "confidence": float(line[1][1])
                })
    results.append({"image": os.path.basename(img_path), "detections": detections, "count": len(detections)})
    if (i + 1) % 50 == 0:
        paddle.device.cuda.empty_cache()
    if (i+1) % 100 == 0:
        elapsed = time.time() - start_time
        print(f"  ✅ {i+1}/{total} | {(i+1)/elapsed:.1f} img/sec", end="\r")

elapsed = time.time() - start_time
print(f"\n{elapsed:.2f} сек ({total/elapsed:.1f} img/sec)")

def save_json(data):
    with open(os.path.join(OUTPUT_FOLDER, Path(data["image"]).stem + ".json"), 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

with ThreadPoolExecutor(max_workers=4) as ex:
    ex.map(save_json, results)

print(f"Save {len(results)} JSON in {OUTPUT_FOLDER}")

Find images: 3630

Running OCR
  ✅ 3600/3630 | 1.2 img/sec
2954.43 сек (1.2 img/sec)
Save 3630 JSON in /kaggle/working/ocr_results


In [6]:
import shutil
from pathlib import Path

output_dir = Path("/kaggle/working/ocr_results")
zip_path = Path("/kaggle/working/ocr_results.zip")

if zip_path.exists(): zip_path.unlink()

!cd /kaggle/working && zip -r -0 ocr_results.zip ocr_results/

print(f"Archive {zip_path} ({zip_path.stat().st_size / 1024**2:.1f} MB)")

  adding: ocr_results/ (stored 0%)
  adding: ocr_results/03316.json (stored 0%)
  adding: ocr_results/00681.json (stored 0%)
  adding: ocr_results/02043.json (stored 0%)
  adding: ocr_results/01018.json (stored 0%)
  adding: ocr_results/02508.json (stored 0%)
  adding: ocr_results/02534.json (stored 0%)
  adding: ocr_results/00366.json (stored 0%)
  adding: ocr_results/00545.json (stored 0%)
  adding: ocr_results/01055.json (stored 0%)
  adding: ocr_results/00970.json (stored 0%)
  adding: ocr_results/02811.json (stored 0%)
  adding: ocr_results/02820.json (stored 0%)
  adding: ocr_results/01585.json (stored 0%)
  adding: ocr_results/03358.json (stored 0%)
  adding: ocr_results/01871.json (stored 0%)
  adding: ocr_results/01662.json (stored 0%)
  adding: ocr_results/00408.json (stored 0%)
  adding: ocr_results/00666.json (stored 0%)
  adding: ocr_results/02842.json (stored 0%)
  adding: ocr_results/02870.json (stored 0%)
  adding: ocr_results/03435.json (stored 0%)
  adding: ocr_result